In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

CWD = Path.cwd()
PROJECT_ROOT = CWD if (CWD / "data").exists() else CWD.parent
PROCESSED = PROJECT_ROOT / "data" / "processed"

points = pd.read_parquet(PROCESSED / "points_parsed.parquet")
shots = pd.read_parquet(PROCESSED / "shots_parsed.parquet")
matches = pd.read_parquet(PROCESSED / "matches_raw.parquet")

print("Points:", points.shape)
print("Shots: ", shots.shape)

Points: (1353711, 10)
Shots:  (5308401, 11)


In [2]:
# Eight match_ids appear twice. Keeping both would duplicate rows on the
# join, so we keep the first occurrence of each.
matches = matches.drop_duplicates(subset="match_id", keep="first")

# Hand columns contain stray values (dates, tournament names) from
# misaligned source rows. Only R and L are valid; everything else is unknown.
for col in ["Pl 1 hand", "Pl 2 hand"]:
    matches[col] = matches[col].astype(str).str.strip()
    matches.loc[~matches[col].isin(["R", "L"]), col] = np.nan

# Same problem in Surface.
valid_surfaces = ["Hard", "Clay", "Grass", "Carpet"]
matches["Surface"] = matches["Surface"].astype(str).str.strip()
matches.loc[~matches["Surface"].isin(valid_surfaces), "Surface"] = np.nan

print(matches["Surface"].value_counts(dropna=False))
print()
print("Left-handed share (P1):", (matches["Pl 1 hand"] == "L").mean().round(3))

Surface
Hard     7592
Clay     2752
Grass    1283
NaN        11
Name: count, dtype: int64

Left-handed share (P1): 0.101


In [3]:
# Keep only the match-level columns we actually need.
match_info = matches[["match_id", "Surface", "Tournament", "Round", "Date",
                      "Pl 1 hand", "Pl 2 hand"]].copy()
match_info = match_info.rename(columns={"Pl 1 hand": "hand_p1",
                                        "Pl 2 hand": "hand_p2"})

points = points.merge(match_info, on="match_id", how="left")

# A left join must not change the number of rows. If it does, the match
# table still contains duplicates and the result cannot be trusted.
print("Points after join:", points.shape)
print("Surface missing after join (%):",
      round(points["Surface"].isna().mean() * 100, 2))

Points after join: (1353711, 16)
Surface missing after join (%): 0.11


In [4]:
# The Pts column is written from the server's perspective: "30-40" means
# the server has 30 and the returner has 40.
BREAK_POINTS = {"0-40", "15-40", "30-40", "40-AD"}
GAME_POINTS = {"40-0", "40-15", "40-30", "AD-40"}
STANDARD_SCORES = {"0", "15", "30", "40", "AD"}


def is_tiebreak(score):
    """Tiebreaks are counted 0,1,2,... so any token outside the normal
    tennis vocabulary means we are in a tiebreak."""
    parts = str(score).split("-")
    return len(parts) == 2 and not all(part in STANDARD_SCORES for part in parts)


points["is_tiebreak"] = points["Pts"].map(is_tiebreak)
points["is_break_point"] = points["Pts"].isin(BREAK_POINTS) & ~points["is_tiebreak"]
points["is_game_point"] = points["Pts"].isin(GAME_POINTS) & ~points["is_tiebreak"]
points["is_deuce"] = (points["Pts"] == "40-40")

# A single label for SQ4, so pressure can be used as one categorical feature.
points["pressure_context"] = np.select(
    [points["is_break_point"], points["is_game_point"],
     points["is_deuce"], points["is_tiebreak"]],
    ["break_point", "game_point", "deuce", "tiebreak"],
    default="regular")

print(points["pressure_context"].value_counts())

pressure_context
regular        842700
game_point     250213
break_point    124827
deuce           96929
tiebreak        39042
Name: count, dtype: int64


In [5]:
# Direction code 1 names the side that is a right-hander's forehand.
# For a left-handed receiver that same side is their backhand, so the
# wing a shot is hit to depends on who is receiving it.
hands = points[["match_id", "hand_p1", "hand_p2"]].drop_duplicates("match_id")
shots = shots.merge(hands, on="match_id", how="left")

# The player receiving shot N is the one who did not hit it.
receiver_hand = np.where(shots["hitter"] == 1, shots["hand_p2"], shots["hand_p1"])

shots["to_opponent_wing"] = np.select(
    [shots["direction"] == "2",
     (shots["direction"] == "1") & (receiver_hand == "R"),
     (shots["direction"] == "3") & (receiver_hand == "L"),
     (shots["direction"] == "3") & (receiver_hand == "R"),
     (shots["direction"] == "1") & (receiver_hand == "L")],
    ["middle", "forehand", "forehand", "backhand", "backhand"],
    default=None)

print(pd.Series(shots["to_opponent_wing"]).value_counts(dropna=False))

to_opponent_wing
backhand    1960119
middle      1745028
forehand    1483276
None         119978
Name: count, dtype: int64


In [6]:
points.to_parquet(PROCESSED / "points_context.parquet", index=False)
shots.to_parquet(PROCESSED / "shots_context.parquet", index=False)
print("Saved.")

Saved.
